# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tahir-MD/FlyRank-Week-01/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding: What Predicts Health? (Random Forest feature importance, ML appendix page 27)**

The paper trains a Random Forest to predict health score, and average position, impressions, and scroll depth come out as the top features. My question is where the label comes from. Health score is built from impressions, position, CTR, and scroll depth added together. Three of the four top predicted features are pieces of the label itself. That means the model may be reconstructing the scoring formula rather than finding an independent pattern. The paper actually says this itself in one line, that importance here is descriptive rather than causal, and I think that caution is fair and worth keeping front and center rather than as a small note.

**Finding: What Predicts Growth? (Logistic regression, 71% holdout accuracy, ML appendix page 29)**

This model predicts growing versus declining pages using signals like content age, days since update, and days visible, and reports 71% holdout accuracy across a sample pulled from 57 brands. My question is about the split design. The paper does not say whether the holdout was grouped by brand or just a random row split. If pages from the same brand can land in both the train set and the test set, the model could be partly learning brand level habits instead of a general growth pattern, and the 71% number would look stronger than it really is. This is not a small detail, since I found this exact gap in my own week five model before I fixed it, so I would ask the same question here that I now ask of myself.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

I re ran my week five Random Forest model two ways on the same data and the same features. The before version uses a plain random row split, where rows from the same client can appear in both the train set and the test set. The after version groups by client id, so no client appears in both sides, which is the same grouped split I already used in week five.

The random row split shows a noticeably higher precision at fifty than the grouped split, and I checked directly, almost every client in the training rows of the random split also shows up in the test rows. That overlap is the reason the random split looks stronger. The grouped split has zero client overlap between train and test, so its lower number is the more honest one, and it is the number I am keeping as my real result going forward.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

path = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
if not os.path.exists(path):
    path = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(path)
df["label"] = (df["trend_direction"] == "down").astype(int)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]
categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

X_num = df[numeric_features].apply(pd.to_numeric, errors="coerce").fillna(0)
X_cat = pd.get_dummies(df[categorical_features].astype(str), dummy_na=False)
X = pd.concat([X_num, X_cat], axis=1)
y = df["label"]

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": y_true.values, "score": scores})
    top = frame.sort_values("score", ascending=False).head(k)
    return top["y"].mean()

def make_model():
    return RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=25,
        class_weight="balanced_subsample", n_jobs=-1, random_state=42
    )

X_train_before, X_test_before, y_train_before, y_test_before = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model_before = make_model()
model_before.fit(X_train_before, y_train_before)
proba_before = model_before.predict_proba(X_test_before)[:, 1]

client_overlap_before = df.loc[X_train_before.index, "client_id"].isin(
    df.loc[X_test_before.index, "client_id"]
).mean()

clients = df["client_id"].astype(str)
unique_clients = clients.unique()
rng = np.random.default_rng(42)
shuffled_clients = rng.permutation(unique_clients)
test_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_count])
test_mask = clients.isin(test_clients).to_numpy()

X_train_after, X_test_after = X[~test_mask], X[test_mask]
y_train_after, y_test_after = y[~test_mask], y[test_mask]
model_after = make_model()
model_after.fit(X_train_after, y_train_after)
proba_after = model_after.predict_proba(X_test_after)[:, 1]

client_overlap_after = len(
    set(df.loc[~test_mask, "client_id"]) & set(df.loc[test_mask, "client_id"])
)

before_after_table = pd.DataFrame([
    {
        "split": "before, random row split",
        "precision_at_50": round(precision_at_k(y_test_before, proba_before, 50), 3),
        "roc_auc": round(roc_auc_score(y_test_before, proba_before), 3),
        "train_rows_with_client_also_in_test": round(client_overlap_before, 3),
    },
    {
        "split": "after, grouped by client",
        "precision_at_50": round(precision_at_k(y_test_after, proba_after, 50), 3),
        "roc_auc": round(roc_auc_score(y_test_after, proba_after), 3),
        "train_rows_with_client_also_in_test": client_overlap_after,
    },
])

before_after_table


,split,precision_at_50,roc_auc,train_rows_with_client_also_in_test
0,"before, random row split",0.92,0.758,1.0
1,"after, grouped by client",0.70,0.749,0.0


## 3. Leakage audit

I ran the same kind of hunt from week three on the final feature list I actually used in week five. First, I checked every numeric feature against the label using a simple correlation, since a feature that is basically a copy of the label would show up as a value close to one. Second, I confirmed that the columns I know are tied to the label, trend direction and trend percent, along with identity columns like client id, provider used, and model used, never made it into the feature list at all.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
label_correlation = df[numeric_features + ["label"]].corr(numeric_only=True)["label"].drop("label")
label_correlation = label_correlation.reindex(label_correlation.abs().sort_values(ascending=False).index)
print(label_correlation.head(10))

excluded_columns = ["trend_direction", "trend_pct", "client_id", "provider_used", "model_used", "content_id"]
used_columns = numeric_features + categorical_features
overlap = set(excluded_columns) & set(used_columns)
print("excluded columns that leaked into the feature list:", overlap)


days_with_impressions     0.190055
content_age_days         -0.163882
word_count                0.090157
days_since_last_update    0.081383
char_count                0.072188
ctr                      -0.061911
clicks_90d               -0.039680
engaged_sessions_90d     -0.035402
avg_position             -0.029035
days_with_sessions       -0.025055
Name: label, dtype: float64
excluded columns that leaked into the feature list: set()


## 4. Claim rewrite

My boldest sentence from week five was, Random Forest reads position and traffic history well. That sentence sounds like a settled fact about the model's ability, and it does not carry the split issues or the sample size in mind.

Rewritten in safe language: on the grouped by client split, Random Forest showed the strongest measured precision at fifty and ROC AUC among the three models tested on this dataset, and its top ranked features were days with impressions, recent impressions, average position, and content age. This is an observed, directional result on one dataset and one split design, and it is meant to support a decision on which model to carry forward, not a general claim about what Random Forest can do.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.